Historical preparation/training reproduction. The completed main test is closed. Use a separate work directory for any independent training; do not substitute its new weights into the frozen evaluated service. Private inputs and authorized W&B access are required.


# Real-Abshar Phase 2 — M4 replacement run with model backups

First inspect saved Kaggle Versions/Output for the old M4 run. If no trained weights remain,
this notebook trains a replacement with the original fixed experiment settings.

Attach your private prepared-data input, enable GPU + Internet, and enable Files-only persistence
BEFORE training. Grant access to the Kaggle secret `WANDB_API_KEY`; online W&B is required.
Use the default NEW recovery work folder. Keep `RESTORE_ARTIFACT = None` for the replacement run.

Before spending training time, an artifact upload is confirmed. Then models/metadata are backed up
at epoch 1, every 5 epochs, stopping, completion and final validation selection. Dataset images/masks
and photo galleries stay out of the W&B artifacts. Keep downloads and saved Kaggle outputs as well.

The cohort stays 552 train / 126 validation / 114 test images from 92 / 21 / 19 complete source
families. The test split is absent from the fit YAML and is not evaluated. Target: one `edited_region`
union box per edited image, empty labels for pristine images. See START_HERE.md for full recovery details.


In [ ]:
# 1. Install the two pinned experiment packages; retain Kaggle's PyTorch/CUDA build.
import importlib.metadata
import os
import subprocess
import sys

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['WANDB_DISABLE_CODE'] = 'true'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
                       'ultralytics==8.3.221', 'wandb==0.28.1'])
print({name: importlib.metadata.version(name) for name in ['ultralytics', 'wandb', 'torch']})
print('If another Ultralytics version was previously imported, restart this kernel and run from cell 1.')


In [ ]:
# 2. Recovery paths, required online W&B backup, optional later restore.
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/real_abshar_phase2_m4_recovery_v1')
PREPARED_ROOT = None
WANDB_PROJECT = 'Real-Abshar-Phase2'
WANDB_ENTITY = None
RESTORE_ARTIFACT = None  # Only for a future restore: 'entity/project/real-abshar-m4-backup-ID:vN'.

if not os.environ.get('WANDB_API_KEY'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    except Exception:
        raise RuntimeError('Enable this notebook access to the WANDB_API_KEY Kaggle secret before training.') from None
if not os.environ.get('WANDB_API_KEY'):
    raise RuntimeError('WANDB_API_KEY is empty. Fix the Kaggle secret before training.')
print('W&B key loaded securely. A live backup upload will be verified before training starts.')


## 3. Load the embedded experiment code

This writes the exact source used by the experiment into your output folder. Its hashes, the
pretrained checkpoint hash, installed package versions and cohort lock are saved with the results.
You do not need to upload the Python files separately. The fit view verifies train/validation
image pixels, 0/1 masks and normalized YOLO boxes before training.

The fourth module adds W&B model/metadata backups and optional verified restoration.


In [ ]:
# 3. Embedded source and dataset/optional backup preflight.
SOURCES = {'cohort_data.py': '"""Load the reviewed preparation and preserve a complete-family Phase 2 cohort."""\nfrom pathlib import Path, PurePosixPath\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nimport shutil\nimport stat\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nSOURCE_MANIFEST_SHA256 = "8e964394838aba82aa85d8e7b93551a4018f0c1d7ce1e96ade17b625aeee14b8"\nSOURCE_ASSIGNMENT_SHA256 = "bcef2a4e57004f1e66645a9ea8a1d14275c3d8f0c7bbe45625760513623ab7cd"\nCOHORT_UID_SHA256 = "c24eb30ac2269840eb4f92e5ada2870493e7624c211a96cb8791d80ce8f40476"\nCOHORT_NAME = "csi-complete-aligned-families-v1"\nEXPECTED = {"train": {"images": 552, "edited": 460, "pristine": 92, "sources": 92},\n            "val": {"images": 126, "edited": 105, "pristine": 21, "sources": 21},\n            "test": {"images": 114, "edited": 95, "pristine": 19, "sources": 19}}\n\n\ndef sha_file(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef dump(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef write_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fields = fields or list(rows[0])\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_path(root, relative):\n    value = str(relative)\n    p = PurePosixPath(value)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in value or ":" in value:\n        raise ValueError("Unsafe dataset-relative path: " + value)\n    candidate = (Path(root) / value).resolve()\n    if not candidate.is_relative_to(Path(root).resolve()):\n        raise ValueError("Dataset path escapes its root.")\n    return candidate\n\n\ndef filter_complete_families(rows):\n    families = defaultdict(list)\n    for row in rows:\n        families[row["source_id"]].append(row)\n    complete = set()\n    for source, values in families.items():\n        if (len(values) == 6 and {int(r["variant"]) for r in values} == set(range(6))\n                and sum(r["kind"] == "pristine" for r in values) == 1\n                and sum(r["kind"] == "edited" for r in values) == 5):\n            complete.add(source)\n    kept = [row for row in rows if row["source_id"] in complete]\n    omitted = [{"uid": row["uid"], "source_id": row["source_id"], "split": row["split"],\n                "reason": "incomplete_source_family_after_annotation_QC"}\n               for row in rows if row["source_id"] not in complete]\n    return kept, omitted\n\n\ndef split_counts(rows):\n    result = {}\n    for split in ["train", "val", "test"]:\n        part = [r for r in rows if r["split"] == split]\n        result[split] = {"images": len(part), "edited": sum(r["kind"] == "edited" for r in part),\n                         "pristine": sum(r["kind"] == "pristine" for r in part),\n                         "sources": len({r["source_id"] for r in part})}\n    return result\n\n\ndef read_preparation(root):\n    root = Path(root)\n    manifest = root / "manifest.csv"\n    if not manifest.is_file() or sha_file(manifest) != SOURCE_MANIFEST_SHA256:\n        raise ValueError("This is not the exact 864-record prepared dataset reviewed in this chat.")\n    candidates = [root / "protocol_lock.json", root.parent / "protocol_lock.json"]\n    locks = [p for p in candidates if p.is_file()]\n    if not locks or not any(json.loads(p.read_text()).get("assignment_sha256") == SOURCE_ASSIGNMENT_SHA256 for p in locks):\n        raise ValueError("The source preparation split lock does not match the reviewed version.")\n    rows = list(csv.DictReader(manifest.open()))\n    kept, omitted = filter_complete_families(rows)\n    digest = hashlib.sha256(json.dumps([r["uid"] for r in kept], separators=(",", ":")).encode()).hexdigest()\n    if digest != COHORT_UID_SHA256 or split_counts(kept) != EXPECTED:\n        raise ValueError("The complete-family cohort differs from the pre-training review decision.")\n    for key in ["source_id", "group_id", "rgb_sha256", "phash"]:\n        values = defaultdict(set)\n        for row in kept:\n            values[row[key]].add(row["split"])\n        if any(len(splits) > 1 for splits in values.values()):\n            raise ValueError("Cross-split relationship found: " + key)\n    return kept, omitted\n\n\ndef locate_prepared(input_root, working_root, explicit=None):\n    if explicit:\n        root = Path(explicit)\n        read_preparation(root)\n        return root\n    candidates = []\n    roots = [Path(input_root), Path("/kaggle/working/real_abshar_phase2_preparation_v1")]\n    for parent in roots:\n        if parent.exists():\n            for manifest in parent.rglob("manifest.csv"):\n                root = manifest.parent\n                if (root / "images").is_dir() and (root / "labels").is_dir() and (root / "segmentation_masks").is_dir():\n                    try:\n                        read_preparation(root)\n                        candidates.append(root)\n                    except ValueError:\n                        pass\n    candidates = sorted(set(p.resolve() for p in candidates))\n    if candidates:\n        if len(candidates) > 1:\n            raise ValueError("Multiple reviewed prepared folders are attached. Set PREPARED_ROOT explicitly: " + str(candidates))\n        return candidates[0]\n    archives = sorted(p for p in Path(input_root).rglob("*.zip")\n                      if p.name.startswith("Real-Abshar_Phase2_Prepared_Data"))\n    matching = []\n    for path in archives:\n        with zipfile.ZipFile(path) as archive:\n            if "manifest.csv" in archive.namelist() and hashlib.sha256(archive.read("manifest.csv")).hexdigest() == SOURCE_MANIFEST_SHA256:\n                matching.append(path)\n    if len(matching) == 1:\n        destination = Path(working_root) / "unpacked_preparation"\n        destination.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(matching[0]) as archive:\n            for item in archive.infolist():\n                if item.is_dir():\n                    continue\n                if stat.S_ISLNK(item.external_attr >> 16):\n                    raise ValueError("Dataset archives must not contain symbolic links.")\n                target = safe_path(destination, item.filename)\n                if target.suffix.lower() not in {".png", ".txt", ".csv", ".json", ".yaml"}:\n                    raise ValueError("Unexpected file in the prepared-data archive: " + item.filename)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(item) as source, target.open("wb") as output:\n                    shutil.copyfileobj(source, output)\n        read_preparation(destination)\n        return destination\n    raise FileNotFoundError(\n        "Attach Real-Abshar_Phase2_Prepared_Data.zip as a PRIVATE Kaggle input, or attach the saved "\n        "preparation notebook outputs. The small review ZIP and the original CSI download are not the prepared dataset.")\n\n\ndef verify_row(root, row):\n    with Image.open(safe_path(root, row["export_image"])) as opened:\n        image = opened.convert("RGB")\n        image.load()\n    size = (int(row["width"]), int(row["height"]))\n    if image.size != size or hashlib.sha256(str(size).encode() + image.tobytes()).hexdigest() != row["rgb_sha256"]:\n        raise ValueError("Image integrity check failed: " + row["uid"])\n    with Image.open(safe_path(root, row["export_mask"])) as opened:\n        mask = np.asarray(opened, dtype=np.uint8)\n    if mask.shape != (size[1], size[0]) or not set(np.unique(mask)).issubset({0, 1}):\n        raise ValueError("Invalid 0/1 segmentation mask: " + row["uid"])\n    if hashlib.sha256(str(size).encode() + mask.tobytes()).hexdigest() != row["mask_sha256"]:\n        raise ValueError("Mask integrity check failed: " + row["uid"])\n    line = safe_path(root, row["export_detection_label"]).read_text().strip()\n    expected_box = json.loads(row["box_xyxy_exclusive"])\n    if row["kind"] == "pristine":\n        if line or mask.any() or expected_box is not None:\n            raise ValueError("Pristine image has a positive target: " + row["uid"])\n    else:\n        yy, xx = np.where(mask > 0)\n        box = [int(xx.min()), int(yy.min()), int(xx.max()) + 1, int(yy.max()) + 1]\n        if box != expected_box:\n            raise ValueError("Mask and box disagree: " + row["uid"])\n        values = list(map(float, line.split()))\n        x0, y0, x1, y1 = box\n        expected = [(x0 + x1) / (2 * size[0]), (y0 + y1) / (2 * size[1]),\n                    (x1 - x0) / size[0], (y1 - y0) / size[1]]\n        if len(values) != 5 or values[0] != 0 or not np.allclose(values[1:], expected, atol=1e-8):\n            raise ValueError("Incorrect YOLO label: " + row["uid"])\n\n\ndef build_training_view(prepared_root, work_root):\n    prepared_root, work_root = Path(prepared_root), Path(work_root)\n    rows, omitted = read_preparation(prepared_root)\n    cohort_dir = work_root / "cohort"\n    cohort_dir.mkdir(parents=True, exist_ok=True)\n    lock = {"cohort": COHORT_NAME, "source_manifest_sha256": SOURCE_MANIFEST_SHA256,\n            "source_assignment_sha256": SOURCE_ASSIGNMENT_SHA256, "cohort_uid_sha256": COHORT_UID_SHA256,\n            "rule": "retain only one pristine plus all five QC-valid edits per source; preserve existing split",\n            "split_counts": EXPECTED, "selected_before_any_model_training": True}\n    lock_path = cohort_dir / "cohort_lock.json"\n    if lock_path.exists() and json.loads(lock_path.read_text()) != lock:\n        raise ValueError("A different cohort is already locked in this M4 work folder.")\n    dump(lock_path, lock)\n    write_csv(cohort_dir / "manifest.csv", rows)\n    write_csv(cohort_dir / "valid_records_omitted_by_cohort_rule.csv", omitted)\n    for split in ["train", "val", "test"]:\n        write_csv(cohort_dir / f"{split}_manifest.csv", [r for r in rows if r["split"] == split])\n    view = work_root / "data_view"\n    fit_rows = [r for r in rows if r["split"] in {"train", "val"}]\n    for index, row in enumerate(fit_rows, 1):\n        verify_row(prepared_root, row)\n        for name in ["export_image", "export_detection_label"]:\n            target = safe_path(view, row[name])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(safe_path(prepared_root, row[name]), target)\n        if index % 100 == 0 or index == len(fit_rows):\n            print(f"Verified/copied train+val {index}/{len(fit_rows)}")\n    data_yaml = view / "data.yaml"\n    data_yaml.write_text("path: " + json.dumps(str(view.resolve()))\n                         + "\\ntrain: images/train\\nval: images/val\\nnames:\\n  0: edited_region\\n")\n    dump(cohort_dir / "preflight.json", {"train_val_records_verified": len(fit_rows),\n         "test_image_or_mask_pixels_read": False, "test_images_copied_to_fit_view": False,\n         "original_preparation_records": 864, "cohort_records": len(rows),\n         "valid_records_omitted_by_complete_family_rule": len(omitted), "split_counts": EXPECTED})\n    return rows, data_yaml\n', 'detection_metrics.py': '"""Operating-point metrics for one edited-region class and at most one GT box per image."""\nfrom collections import Counter\nimport json\nimport math\n\n\ndef box_iou(a, b):\n    x0, y0 = max(a[0], b[0]), max(a[1], b[1])\n    x1, y1 = min(a[2], b[2]), min(a[3], b[3])\n    intersection = max(0.0, x1 - x0) * max(0.0, y1 - y0)\n    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])\n    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])\n    denominator = area_a + area_b - intersection\n    return intersection / denominator if denominator else 0.0\n\n\ndef operating_metrics(rows, predictions, confidence, matching_iou=0.5, input_size=640):\n    totals = Counter()\n    per_image = []\n    sizes = {name: Counter(instances=0, matched=0) for name in ["small", "medium", "large"]}\n    for row in rows:\n        gt = json.loads(row["box_xyxy_exclusive"])\n        boxes = sorted((p for p in predictions.get(row["uid"], []) if float(p["score"]) >= confidence),\n                       key=lambda p: (-float(p["score"]), p["box"]))\n        local = Counter(tp=0, fp=0, fn=0, duplicate=0, localisation=0, background=0)\n        matched = False\n        for prediction in boxes:\n            box, score = prediction["box"], float(prediction["score"])\n            if len(box) != 4 or not all(math.isfinite(float(v)) for v in box) or not math.isfinite(score):\n                raise ValueError("Non-finite or malformed prediction.")\n            if int(prediction.get("class", 0)) != 0:\n                raise ValueError("This evaluator expects the single edited_region class.")\n            iou = box_iou(box, gt) if gt is not None else 0.0\n            if gt is not None and iou >= matching_iou and not matched:\n                local["tp"] += 1\n                matched = True\n            else:\n                local["fp"] += 1\n                local["duplicate" if gt is not None and iou >= matching_iou else\n                      "localisation" if gt is not None and iou >= 0.1 else "background"] += 1\n        if gt is not None:\n            local["fn"] = int(not matched)\n            scale = input_size / max(int(row["width"]), int(row["height"]))\n            area = (gt[2] - gt[0]) * (gt[3] - gt[1]) * scale * scale\n            bin_name = "small" if area < 32 ** 2 else "medium" if area < 96 ** 2 else "large"\n            sizes[bin_name]["instances"] += 1\n            sizes[bin_name]["matched"] += int(matched)\n        totals.update(local)\n        per_image.append({"uid": row["uid"], "source_id": row["source_id"], "kind": row["kind"],\n                          "manipulation": row.get("manipulation", ""), **local})\n    tp, fp, fn = totals["tp"], totals["fp"], totals["fn"]\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    result = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1,\n              "confidence": confidence, "matching_iou": matching_iou,\n              "duplicate_fp": totals["duplicate"], "localisation_fp": totals["localisation"],\n              "background_fp": totals["background"], "images": len(rows),\n              "recall_by_size": {name: {**value, "recall": value["matched"] / value["instances"]\n                                 if value["instances"] else None} for name, value in sizes.items()}}\n    return result, per_image\n\n\ndef threshold_sweep(rows, caches, input_size=640):\n    values = []\n    thresholds = [0.01, 0.025] + [round(i * 0.05, 3) for i in range(1, 20)]\n    for nms_iou, predictions in sorted(caches.items()):\n        for confidence in thresholds:\n            metrics, _ = operating_metrics(rows, predictions, confidence, input_size=input_size)\n            values.append({key: metrics[key] for key in ["tp", "fp", "fn", "precision", "recall", "f1", "confidence"]}\n                          | {"nms_iou": nms_iou, "split": "val", "input_size": input_size})\n    selected = max(values, key=lambda r: (r["f1"], r["recall"], r["precision"], r["confidence"], -r["nms_iou"]))\n    return selected, values\n', 'm4_train.py': '"""Kaggle M4 experiment: train two detectors and select on validation only."""\nfrom pathlib import Path\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport platform\nimport shutil\nimport time\nimport urllib.request\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nfrom cohort_data import (COHORT_NAME, COHORT_UID_SHA256, EXPECTED, build_training_view,\n                         dump, locate_prepared, safe_path, sha_file, write_csv)\nfrom detection_metrics import operating_metrics, threshold_sweep\nfrom wandb_backup import BACKUP_EVERY_EPOCHS, upload_backup\n\nULTRALYTICS_VERSION = "8.3.221"\nWANDB_VERSION = "0.28.1"\nPRETRAINED_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt"\nPRETRAINED_BYTES = 5613764\nIMGSZ = 640\nLOW_CONFIDENCE = 0.001\nCOMPARISON_NMS = 0.7\nMAX_DET = 50\nCOMMON = dict(epochs=30, patience=8, imgsz=IMGSZ, batch=8, workers=2, device=0,\n              optimizer="AdamW", lr0=0.001, lrf=0.01, weight_decay=0.0005,\n              nbs=64, seed=42, deterministic=True, amp=False, cache=False, rect=False,\n              cos_lr=False, val=True, save=True, save_period=1, plots=False,\n              conf=LOW_CONFIDENCE, iou=COMPARISON_NMS, max_det=MAX_DET,\n              degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0,\n              mixup=0.0, cutmix=0.0, copy_paste=0.0, fliplr=0.5)\nEXPERIMENTS = {\n    "A_standard_augmentation": dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,\n                                    mosaic=1.0, close_mosaic=10, translate=0.1, scale=0.5),\n    "B_reduced_augmentation": dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,\n                                   mosaic=0.0, close_mosaic=0, translate=0.05, scale=0.1),\n}\n\n\ndef number(value):\n    try:\n        value = float(value)\n        return value if math.isfinite(value) else None\n    except (TypeError, ValueError):\n        return None\n\n\ndef environment():\n    import torch\n    names = ["ultralytics", "wandb", "torch", "torchvision", "numpy", "Pillow",\n             "scipy", "opencv-python", "albumentations", "matplotlib"]\n    versions = {}\n    for name in names:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    if versions["ultralytics"] != ULTRALYTICS_VERSION or versions["wandb"] != WANDB_VERSION:\n        raise RuntimeError("Run the pinned dependency cell in a fresh Kaggle session before importing this module.")\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU accelerator, then restart and run all cells.")\n    return {"python": platform.python_version(), "packages": versions,\n            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),\n            "gpu_device": 0, "test_inference_performed": False}\n\n\ndef initial_checkpoint(work_root):\n    destination = work_root / "pretrained" / "yolo11n.pt"\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if not destination.exists():\n        temporary = destination.with_suffix(".download")\n        with urllib.request.urlopen(PRETRAINED_URL, timeout=60) as source, temporary.open("wb") as output:\n            shutil.copyfileobj(source, output)\n        if temporary.stat().st_size != PRETRAINED_BYTES:\n            temporary.unlink()\n            raise ValueError("The official pretrained asset size changed; review the download before training.")\n        temporary.replace(destination)\n    if destination.stat().st_size != PRETRAINED_BYTES:\n        raise ValueError("The local pretrained checkpoint has an unexpected size.")\n    return destination\n\n\ndef lock_protocol(work_root, pretrained, env):\n    sources = {p.name: sha_file(p) for p in Path(__file__).parent.glob("*.py")}\n    protocol = {"cohort": COHORT_NAME, "cohort_uid_sha256": COHORT_UID_SHA256,\n                "splits": EXPECTED, "architecture": "YOLO11n, one class: edited_region",\n                "pretrained_url": PRETRAINED_URL, "pretrained_sha256": sha_file(pretrained),\n                "common_train_settings": COMMON, "experiments": EXPERIMENTS,\n                "comparison": {"split": "val", "input_size": [IMGSZ, IMGSZ],\n                               "confidence_floor": LOW_CONFIDENCE, "nms_iou": COMPARISON_NMS,\n                               "criterion": "highest validation mAP50-95; tie resolves to run A"},\n                "operating_point": {"split": "val", "matching_iou": 0.5,\n                                    "nms_iou_grid": [0.45, 0.6, 0.75],\n                                    "confidence_grid": [0.01, 0.025] + [round(i * 0.05, 3) for i in range(1, 20)],\n                                    "criterion": "max F1; tie: recall, precision, higher confidence, lower NMS IoU"},\n                "source_sha256": sources,\n                "run_purpose": "replacement_after_unsaved_session_outputs",\n                "checkpoint_backup": {"backend": "W&B Artifacts", "every_epochs": BACKUP_EVERY_EPOCHS,\n                                      "also": ["preflight", "epoch_1", "stopping_epoch", "run_complete", "M4_complete"],\n                                      "dataset_images_or_masks_uploaded": False},\n                "package_versions": env["packages"], "python": env["python"],\n                "test_evaluation_authorized_in_this_notebook": False}\n    path = work_root / "m4_protocol_lock.json"\n    if path.exists() and json.loads(path.read_text()) != protocol:\n        raise ValueError("This work folder has a different locked protocol. Restore its original files/environment or use a new folder.")\n    dump(path, protocol)\n    dump(work_root / "environment.json", env)\n    return protocol\n\n\ndef start_wandb(work_root, project, entity, name, configuration):\n    import wandb\n    online = bool(os.environ.get("WANDB_API_KEY"))\n    if not online:\n        raise RuntimeError("Recovery training requires an online WANDB_API_KEY secret so checkpoints can be backed up.")\n    ids = work_root / "wandb_ids"\n    ids.mkdir(parents=True, exist_ok=True)\n    path = ids / (name + ".txt")\n    if not path.exists():\n        path.write_text(wandb.util.generate_id())\n    kwargs = dict(project=project, name=name, id=path.read_text().strip(),\n                  config=configuration, mode="online" if online else "offline",\n                  dir=str(work_root), save_code=False,\n                  tags=["phase2", "M4", "replacement-after-session-output-loss"])\n    if entity:\n        kwargs["entity"] = entity\n    if online:\n        kwargs["resume"] = "allow"\n    # Manual logging prevents duplicate runs from Ultralytics\' optional integration.\n    run = wandb.init(**kwargs)\n    dump(work_root / "wandb_ids" / (name + ".json"),\n         {"mode": "online" if online else "offline", "project": project,\n          "entity": entity, "id": run.id, "url": run.url if online else None})\n    return run\n\n\ndef free_gpu():\n    import torch\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef train_experiment(work_root, data_yaml, pretrained, name, augmentation, project, entity, protocol):\n    from ultralytics import YOLO\n    directory = work_root / "runs" / name\n    best = directory / "weights" / "best.pt"\n    last = directory / "weights" / "last.pt"\n    marker = directory / "training_complete.json"\n    if marker.exists():\n        previous = json.loads(marker.read_text())\n        if best.is_file() and previous["best_sha256"] == sha_file(best):\n            print("Reusing completed training:", name)\n            return best\n        raise ValueError("A completed run\'s best checkpoint changed: " + name)\n    if directory.exists() and not last.exists() and (directory / "results.csv").exists():\n        raise RuntimeError("Partial run has no last.pt to resume. Retain its outputs and use a fresh WORK_ROOT.")\n    run = start_wandb(work_root, project, entity, name,\n                     {"cohort": COHORT_NAME, "split_counts": EXPECTED, "settings": COMMON | augmentation,\n                      "pretrained_sha256": protocol["pretrained_sha256"], "test_used": False})\n    model = YOLO(str(last if last.is_file() else pretrained))\n\n    def log_epoch(trainer):\n        values = {"epoch": int(trainer.epoch) + 1}\n        for key, value in getattr(trainer, "metrics", {}).items():\n            numeric = number(value)\n            if numeric is not None:\n                values[key] = numeric\n        run.log(values)\n\n    model.add_callback("on_fit_epoch_end", log_epoch)\n\n    def back_up_checkpoint(trainer):\n        from wandb_backup import should_backup\n        epoch = int(trainer.epoch) + 1\n        if should_backup(epoch, bool(getattr(trainer, "stop", False))):\n            upload_backup(run, work_root, f"{name}_epoch_{epoch}")\n\n    model.add_callback("on_model_save", back_up_checkpoint)\n    started = time.monotonic()\n    try:\n        upload_backup(run, work_root, name + "_preflight")\n        if last.is_file():\n            print("Resuming saved checkpoint:", name)\n            try:\n                model.train(resume=True)\n            except AssertionError as error:\n                # A session may end after Ultralytics strips the final optimizer but before our marker is saved.\n                if "nothing to resume" not in str(error).lower() or not best.is_file() or not (directory / "results.csv").is_file():\n                    raise\n                print("Checkpoint already completed training; retaining its best.pt.")\n        else:\n            model.train(data=str(data_yaml), project=str(work_root / "runs"), name=name,\n                        exist_ok=True, **COMMON, **augmentation)\n        if not best.is_file():\n            raise FileNotFoundError("Training did not produce best.pt for " + name)\n        completed_epochs = len(list(csv.DictReader((directory / "results.csv").open())))\n        metadata = {"run": name, "best_sha256": sha_file(best), "completed_epochs": completed_epochs,\n                    "checkpoint_selection": "Ultralytics validation fitness within this run",\n                    "session_training_seconds": time.monotonic() - started, "test_used": False}\n        dump(marker, metadata)\n        run.summary.update(metadata)\n        upload_backup(run, work_root, name + "_complete")\n    finally:\n        run.finish()\n        del model\n        free_gpu()\n    return best\n\n\ndef validate_checkpoint(checkpoint, data_yaml, work_root, name, nms_iou):\n    from ultralytics import YOLO\n    model = YOLO(str(checkpoint))\n    metrics = model.val(data=str(data_yaml), split="val", imgsz=IMGSZ, batch=8, workers=2,\n                        device=0, half=False, rect=False, conf=LOW_CONFIDENCE,\n                        iou=nms_iou, max_det=MAX_DET, plots=False, save_json=False,\n                        project=str(work_root / "validation"), name=name, exist_ok=True, verbose=False)\n    result = {"split": "val", "input_width": IMGSZ, "input_height": IMGSZ,\n              "preprocessing": "aspect-preserving letterbox to 640x640; native-coordinate scoring",\n              "confidence_floor": LOW_CONFIDENCE, "nms_iou": nms_iou, "max_det": MAX_DET,\n              "mAP50_95": float(metrics.box.map), "mAP50": float(metrics.box.map50),\n              "checkpoint_sha256": sha_file(checkpoint)}\n    dump(work_root / "metrics" / (name + ".json"), result)\n    if name == "selected_detector_val_AP":\n        curve = metrics.box.curves_results[0]\n        recall, precision = np.asarray(curve[0]), np.asarray(curve[1])\n        if precision.size:\n            precision = precision.reshape(-1, len(recall))[0]\n            write_csv(work_root / "metrics" / "val_selected_PR.csv",\n                      [{"recall": float(x), "precision": float(y)} for x, y in zip(recall, precision, strict=True)])\n    del model, metrics\n    free_gpu()\n    return result\n\n\ndef validation_predictions(checkpoint, rows, work_root, nms_iou):\n    from ultralytics import YOLO\n    path = work_root / "validation_predictions" / f"val_nms_{nms_iou:.2f}.json"\n    signature = {"split": "val", "input_size": [IMGSZ, IMGSZ], "confidence_floor": LOW_CONFIDENCE,\n                 "nms_iou": nms_iou, "max_det": MAX_DET, "checkpoint_sha256": sha_file(checkpoint),\n                 "cohort_uid_sha256": COHORT_UID_SHA256}\n    if path.exists():\n        cache = json.loads(path.read_text())\n        if cache["settings"] != signature or set(cache["predictions"]) != {r["uid"] for r in rows}:\n            raise ValueError("The saved validation predictions belong to different settings.")\n        return cache["predictions"]\n    model = YOLO(str(checkpoint))\n    paths = [str(safe_path(work_root / "data_view", r["export_image"])) for r in rows]\n    predictions = {}\n    for row, result in zip(rows, model.predict(source=paths, stream=True, batch=8, device=0,\n                                             imgsz=IMGSZ, rect=False, half=False,\n                                             conf=LOW_CONFIDENCE, iou=nms_iou, max_det=MAX_DET,\n                                             save=False, verbose=False), strict=True):\n        if Path(result.path).stem != row["uid"]:\n            raise ValueError("Prediction image order changed; refusing to mispair predictions.")\n        if tuple(result.orig_shape) != (int(row["height"]), int(row["width"])):\n            raise ValueError("Prediction dimensions differ from the manifest.")\n        predictions[row["uid"]] = [{"box": [float(v) for v in box], "score": float(score), "class": int(cls)}\n                                   for box, score, cls in zip(result.boxes.xyxy.cpu().tolist(),\n                                                               result.boxes.conf.cpu().tolist(),\n                                                               result.boxes.cls.cpu().tolist(), strict=True)]\n    dump(path, {"settings": signature, "predictions": predictions})\n    del model\n    free_gpu()\n    return predictions\n\n\ndef make_figures(work_root, rows, comparison, grid, selection, predictions, per_image):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Rectangle\n    figures = work_root / "figures"\n    figures.mkdir(exist_ok=True)\n    fig, ax = plt.subplots(figsize=(9, 5))\n    x = np.arange(len(comparison))\n    ax.bar(x - 0.17, [r["mAP50_95"] for r in comparison], width=0.34, label="mAP50–95")\n    ax.bar(x + 0.17, [r["mAP50"] for r in comparison], width=0.34, label="mAP50")\n    ax.set_xticks(x, [r["run"].replace("_", " ") for r in comparison], fontsize=9)\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Average precision")\n    ax.legend()\n    ax.set_title("Validation • letterbox 640×640 • confidence floor 0.001\\nNMS IoU 0.70 • AP matching IoU 0.50:0.95 • max detections 50", fontsize=10)\n    fig.tight_layout()\n    fig.savefig(figures / "val_detector_comparison.png", dpi=160)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(9, 5))\n    for nms in [0.45, 0.6, 0.75]:\n        part = [r for r in grid if r["nms_iou"] == nms]\n        ax.plot([r["confidence"] for r in part], [r["f1"] for r in part], marker=".", label=f"NMS IoU {nms:.2f}")\n    ax.scatter([selection["confidence"]], [selection["f1"]], marker="*", s=150, color="black", zorder=5, label="Selected")\n    ax.set(xlabel="Confidence threshold (sweep 0.01, 0.025, 0.05…0.95)", ylabel="Detection F1", ylim=(0, 1))\n    ax.set_title("Validation • letterbox 640×640 • matching IoU ≥ 0.50\\nConfidence and NMS thresholds selected jointly; max detections 50", fontsize=10)\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(figures / "val_threshold_sweep.png", dpi=160)\n    plt.close(fig)\n\n    pr_path = work_root / "metrics" / "val_selected_PR.csv"\n    if pr_path.exists():\n        pr = list(csv.DictReader(pr_path.open()))\n        fig, ax = plt.subplots(figsize=(7, 5))\n        ax.plot([float(r["recall"]) for r in pr], [float(r["precision"]) for r in pr])\n        ax.set(xlabel="Recall", ylabel="Interpolated precision", xlim=(0, 1), ylim=(0, 1))\n        ax.set_title(f\'Validation PR at matching IoU 0.50 • letterbox 640×640\\n\'\n                     f\'Confidence floor 0.001 • NMS IoU {selection["nms_iou"]:.2f} • max detections 50\', fontsize=10)\n        fig.tight_layout()\n        fig.savefig(figures / "val_selected_PR.png", dpi=160)\n        plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    for item in comparison:\n        with (work_root / "runs" / item["run"] / "results.csv").open() as stream:\n            history = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(stream)]\n        epochs = [float(r["epoch"]) for r in history]\n        short_name = item["run"][0]\n        axes[0].plot(epochs, [float(r["train/box_loss"]) for r in history], label=f\'{short_name} train\')\n        axes[0].plot(epochs, [float(r["val/box_loss"]) for r in history], linestyle="--", label=f\'{short_name} val\')\n        axes[1].plot(epochs, [float(r["metrics/mAP50-95(B)"]) for r in history], label=short_name)\n    axes[0].set(xlabel="Epoch", ylabel="Recorded box regression loss")\n    axes[1].set(xlabel="Epoch", ylabel="Validation mAP50–95", ylim=(0, 1))\n    for ax in axes:\n        ax.legend()\n    fig.suptitle(\'Train/validation learning curves • letterbox 640×640\\n\'\n                 \'Validation confidence floor 0.001 • NMS IoU 0.70 • AP matching IoU 0.50:0.95\', fontsize=10)\n    fig.tight_layout()\n    fig.savefig(figures / "train_val_learning_curves.png", dpi=160)\n    plt.close(fig)\n\n    # Deterministic validation examples; this is a development gallery, not the final test gallery.\n    candidates = []\n    for rule in [lambda r: r["fn"] > 0, lambda r: r["fp"] > 0,\n                 lambda r: r["tp"] > 0, lambda r: r["kind"] == "pristine" and r["fp"] == 0]:\n        subset = sorted((r for r in per_image if rule(r)), key=lambda r: (-r["fn"], -r["fp"], r["uid"]))\n        candidates.extend([r["uid"] for r in subset[:2]])\n    uids = list(dict.fromkeys(candidates))\n    uids += [r["uid"] for r in per_image if r["uid"] not in uids]\n    uids = uids[:8]\n    by_uid = {r["uid"]: r for r in rows}\n    count_by_uid = {r["uid"]: r for r in per_image}\n    fig, axes = plt.subplots(4, 2, figsize=(12, 17))\n    for ax, uid in zip(axes.flat, uids, strict=True):\n        row = by_uid[uid]\n        with Image.open(safe_path(work_root / "data_view", row["export_image"])) as opened:\n            ax.imshow(opened.convert("RGB"))\n        gt = json.loads(row["box_xyxy_exclusive"])\n        if gt is not None:\n            x0, y0, x1, y1 = gt\n            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="#00e070", linewidth=2))\n        for prediction in predictions[uid]:\n            if prediction["score"] < selection["confidence"]:\n                continue\n            x0, y0, x1, y1 = prediction["box"]\n            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="#ff9e24", linewidth=1.7))\n            ax.text(x0, y0, f\'{prediction["score"]:.2f}\', color="#ff9e24", fontsize=8,\n                    bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))\n        counts = count_by_uid[uid]\n        ax.set_title(f\'{uid} • TP {counts["tp"]}, FP {counts["fp"]}, FN {counts["fn"]}\', fontsize=9)\n        ax.axis("off")\n    fig.suptitle(f\'Validation development gallery • letterbox {IMGSZ}×{IMGSZ}\\n\'\n                 f\'Confidence ≥ {selection["confidence"]:.3f} • NMS IoU {selection["nms_iou"]:.2f} • matching IoU ≥ 0.50\\n\'\n                 \'Green: ground-truth union region; orange: predicted region; coordinates shown on native images\', fontsize=11)\n    fig.tight_layout(rect=(0, 0, 1, 0.94))\n    fig.savefig(figures / "val_development_gallery.png", dpi=130)\n    plt.close(fig)\n\n\ndef bundle_outputs(work_root, selected_checkpoint, selection):\n    output = work_root.parent\n    review_path = output / "Real-Abshar_Phase2_M4_Results_Review.zip"\n    models_path = output / "Real-Abshar_Phase2_M4_Models.zip"\n    with zipfile.ZipFile(review_path, "w", zipfile.ZIP_DEFLATED) as archive:\n        for folder in ["cohort", "metrics", "figures", "validation_predictions", "wandb_ids", "source"]:\n            for path in sorted((work_root / folder).rglob("*")):\n                if path.is_file() and "__pycache__" not in path.parts:\n                    archive.write(path, path.relative_to(work_root).as_posix())\n        for filename in ["m4_protocol_lock.json", "environment.json", "m4_selection.json", "cloud_backup_receipt.json"]:\n            archive.write(work_root / filename, filename)\n        for name in EXPERIMENTS:\n            for filename in ["args.yaml", "results.csv", "training_complete.json"]:\n                path = work_root / "runs" / name / filename\n                if path.exists():\n                    archive.write(path, f"runs/{name}/{filename}")\n    with zipfile.ZipFile(models_path, "w", zipfile.ZIP_DEFLATED) as archive:\n        archive.write(work_root / "pretrained" / "yolo11n.pt", "pretrained/yolo11n.pt")\n        for name in EXPERIMENTS:\n            archive.write(work_root / "runs" / name / "weights" / "best.pt", f"runs/{name}/best.pt")\n        archive.write(selected_checkpoint, "selected/detector_best.pt")\n        for filename in ["m4_protocol_lock.json", "environment.json", "m4_selection.json", "cloud_backup_receipt.json"]:\n            archive.write(work_root / filename, filename)\n        for path in sorted((work_root / "source").glob("*.py")):\n            archive.write(path, "source/" + path.name)\n    dump(work_root / "output_archives.json", {"review": {"file": review_path.name, "sha256": sha_file(review_path)},\n         "models": {"file": models_path.name, "sha256": sha_file(models_path)},\n         "note": "Keep both archives; send the small Results_Review archive for review. Test inference was not run."})\n    return review_path, models_path\n\n\ndef run(input_root="/kaggle/input", work_root="/kaggle/working/real_abshar_phase2_m4_recovery_v1",\n        prepared_root=None, wandb_project="Real-Abshar-Phase2", wandb_entity=None):\n    import ultralytics\n    # Manual W&B logging plus curated model/metadata artifacts; dataset images/masks stay in Kaggle.\n    ultralytics.settings.update({"wandb": False})\n    work_root = Path(work_root)\n    work_root.mkdir(parents=True, exist_ok=True)\n    env = environment()\n    root = locate_prepared(input_root, work_root, prepared_root)\n    print("Prepared input:", root)\n    rows, data_yaml = build_training_view(root, work_root)\n    pretrained = initial_checkpoint(work_root)\n    protocol = lock_protocol(work_root, pretrained, env)\n    checkpoints, comparisons = {}, []\n    for name, augmentation in EXPERIMENTS.items():\n        checkpoint = train_experiment(work_root, data_yaml, pretrained, name, augmentation,\n                                      wandb_project, wandb_entity, protocol)\n        checkpoints[name] = checkpoint\n        metrics = validate_checkpoint(checkpoint, data_yaml, work_root, name + "_common_val", COMPARISON_NMS)\n        comparisons.append({"run": name, **metrics})\n    write_csv(work_root / "metrics" / "val_run_comparison.csv", comparisons)\n    winner = max(comparisons, key=lambda r: r["mAP50_95"])\n    checkpoint = checkpoints[winner["run"]]\n    print("Validation-selected detector:", winner["run"])\n    validation = [r for r in rows if r["split"] == "val"]\n    caches = {nms: validation_predictions(checkpoint, validation, work_root, nms) for nms in [0.45, 0.6, 0.75]}\n    selection, grid = threshold_sweep(validation, caches, IMGSZ)\n    write_csv(work_root / "metrics" / "val_threshold_sweep.csv", grid)\n    selected_predictions = caches[selection["nms_iou"]]\n    operating, per_image = operating_metrics(validation, selected_predictions, selection["confidence"], input_size=IMGSZ)\n    write_csv(work_root / "metrics" / "val_operating_per_image.csv", per_image)\n    selected_ap = validate_checkpoint(checkpoint, data_yaml, work_root, "selected_detector_val_AP", selection["nms_iou"])\n    final = {"selected_run": winner["run"], "checkpoint_sha256": sha_file(checkpoint),\n             "cohort_uid_sha256": COHORT_UID_SHA256, "selection_split": "val",\n             "input_size": [IMGSZ, IMGSZ], "confidence": selection["confidence"],\n             "nms_iou": selection["nms_iou"], "max_det": MAX_DET,\n             "matching_iou_for_operating_metrics": 0.5,\n             "validation_operating_metrics": operating, "validation_AP": selected_ap,\n             "test_evaluation_performed": False,\n             "limitations": ["132 complete aligned source families; not the full CSI-IMD benchmark",\n                             "one union region per edited image; not semantic object instances",\n                             "one seed per augmentation condition; no multi-seed uncertainty claim",\n                             "M5 segmentation and M8 integration still required"]}\n    dump(work_root / "m4_selection.json", final)\n    make_figures(work_root, validation, comparisons, grid, selection, selected_predictions, per_image)\n    run_record = start_wandb(work_root, wandb_project, wandb_entity, "M4_validation_selection",\n                             {"cohort": COHORT_NAME, "selected_run": winner["run"], "test_used": False})\n    try:\n        run_record.log({"val/mAP50_95": selected_ap["mAP50_95"], "val/mAP50": selected_ap["mAP50"],\n                        "val/precision": operating["precision"], "val/recall": operating["recall"],\n                        "val/f1": operating["f1"], "confidence": selection["confidence"], "nms_iou": selection["nms_iou"]})\n        run_record.summary.update({"selected_run": winner["run"], "test_evaluated": False})\n        upload_backup(run_record, work_root, "M4_completed")\n    finally:\n        run_record.finish()\n    review, models = bundle_outputs(work_root, checkpoint, final)\n    print(json.dumps({"selected_run": winner["run"], "validation_mAP50_95": selected_ap["mAP50_95"],\n                      "validation_F1": operating["f1"], "confidence": selection["confidence"],\n                      "nms_iou": selection["nms_iou"], "test_evaluated": False}, indent=2))\n    print("SEND HERE:", review)\n    print("KEEP FOR SEGMENTATION/INTEGRATION:", models)\n    return final, review, models\n', 'wandb_backup.py': '"""Back up model checkpoints and experiment metadata, without dataset images/masks."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nimport shutil\nimport tempfile\nimport uuid\n\nfrom cohort_data import dump, safe_path, sha_file\n\nBACKUP_EVERY_EPOCHS = 5\nBACKUP_WAIT_SECONDS = 300\n\n\ndef backup_files(work_root):\n    root = Path(work_root)\n    selected = set()\n    for name in ["m4_protocol_lock.json", "environment.json", "m4_selection.json", "backup_session_id.txt"]:\n        path = root / name\n        if path.is_file():\n            selected.add(path)\n    for folder, suffixes in {"cohort": {".csv", ".json"}, "metrics": {".csv", ".json"},\n                             "validation_predictions": {".json"}, "wandb_ids": {".txt", ".json"},\n                             "source": {".py"}}.items():\n        for path in (root / folder).glob("*"):\n            if path.is_file() and path.suffix.lower() in suffixes:\n                selected.add(path)\n    pretrained = root / "pretrained" / "yolo11n.pt"\n    if pretrained.is_file():\n        selected.add(pretrained)\n    for run in ["A_standard_augmentation", "B_reduced_augmentation"]:\n        for relative in ["args.yaml", "results.csv", "training_complete.json", "weights/best.pt", "weights/last.pt"]:\n            path = root / "runs" / run / relative\n            if path.is_file():\n                selected.add(path)\n    # Scientific plots are regenerated from backed-up numbers. No photographs enter the artifact.\n    for path in selected:\n        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):\n            raise ValueError("Backup file escapes its work folder: " + str(path))\n    return sorted(selected)\n\n\ndef should_backup(epoch, stopping=False):\n    return epoch == 1 or epoch % BACKUP_EVERY_EPOCHS == 0 or stopping\n\n\ndef upload_backup(run, work_root, stage):\n    import wandb\n    if not os.environ.get("WANDB_API_KEY") or getattr(run.settings, "mode", None) != "online":\n        raise RuntimeError("Online W&B is required for recovery backups. Enable the WANDB_API_KEY secret and Internet before training.")\n    root = Path(work_root)\n    session = root / "backup_session_id.txt"\n    if not session.exists():\n        session.write_text(uuid.uuid4().hex[:12])\n    paths = backup_files(root)\n    if not paths:\n        raise RuntimeError("No experiment files exist to back up.")\n    manifest = {"schema": "real-abshar-m4-backup-v1", "stage": stage,\n                "created_utc": datetime.now(timezone.utc).isoformat(),\n                "dataset_images_or_masks_included": False,\n                "files": {p.relative_to(root).as_posix(): {"sha256": sha_file(p), "bytes": p.stat().st_size}\n                          for p in paths}}\n    manifest_path = root / "cloud_backup_manifest.json"\n    dump(manifest_path, manifest)\n    artifact = wandb.Artifact("real-abshar-m4-backup-" + session.read_text().strip(), type="model",\n                              metadata={"stage": stage, "dataset_images_included": False,\n                                        "file_count": len(paths), "test_used": False})\n    for path in paths:\n        artifact.add_file(str(path), name=path.relative_to(root).as_posix(), policy="mutable")\n    artifact.add_file(str(manifest_path), name="cloud_backup_manifest.json", policy="mutable")\n    logged = run.log_artifact(artifact, aliases=["latest"])\n    logged.wait(timeout=BACKUP_WAIT_SECONDS)\n    # Publish a receipt only after the SDK confirms the upload completed.\n    receipt = {"stage": stage, "artifact": logged.qualified_name, "url": logged.url,\n               "confirmed_utc": datetime.now(timezone.utc).isoformat(),\n               "file_count": len(paths), "dataset_images_or_masks_included": False}\n    dump(root / "cloud_backup_receipt.json", receipt)\n    run.summary["latest_model_backup"] = receipt["artifact"]\n    print("BACKUP_CONFIRMED:", stage, receipt["artifact"], flush=True)\n    print("Backup link:", receipt["url"], flush=True)\n    return receipt\n\n\ndef restore_backup(reference, destination):\n    """Restore an own-project artifact into its original work path; reject corrupt/conflicting files."""\n    import wandb\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix="m4_restore_", dir=destination.parent) as directory:\n        downloaded = Path(wandb.Api().artifact(reference).download(root=directory))\n        manifest = json.loads((downloaded / "cloud_backup_manifest.json").read_text())\n        if manifest.get("schema") != "real-abshar-m4-backup-v1" or manifest.get("dataset_images_or_masks_included") is not False:\n            raise ValueError("Unexpected model-backup schema.")\n        for name, expected in manifest["files"].items():\n            source, target = safe_path(downloaded, name), safe_path(destination, name)\n            if not source.is_file() or source.stat().st_size != expected["bytes"] or sha_file(source) != expected["sha256"]:\n                raise ValueError("Corrupt backup file: " + name)\n            if target.exists() and (not target.is_file() or sha_file(target) != expected["sha256"]):\n                raise ValueError("Restore would replace a different local file: " + str(target))\n        for name in manifest["files"]:\n            source, target = safe_path(downloaded, name), safe_path(destination, name)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(source, target)\n        shutil.copyfile(downloaded / "cloud_backup_manifest.json", destination / "cloud_backup_manifest.json")\n    print("Restored model/metadata backup:", reference)\n    print("Keep the original WORK_ROOT path and attach the prepared dataset before resuming.")\n    return destination\n'}
SOURCE_DIR = WORK_ROOT / 'source'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for filename, source in SOURCES.items():
    (SOURCE_DIR / filename).write_text(source)
sys.path.insert(0, str(SOURCE_DIR))
for module_name in ['m4_train', 'wandb_backup', 'detection_metrics', 'cohort_data']:
    sys.modules.pop(module_name, None)
from wandb_backup import restore_backup
if RESTORE_ARTIFACT:
    restore_backup(RESTORE_ARTIFACT, WORK_ROOT)
from cohort_data import EXPECTED, read_preparation, locate_prepared
ROOT = locate_prepared(INPUT_ROOT, WORK_ROOT, PREPARED_ROOT)
COHORT, OMITTED = read_preparation(ROOT)
print('Prepared dataset:', ROOT)
print('Frozen cohort:', EXPECTED)
print('Complete-family omissions:', len(OMITTED))
print('Test inference executed: False')


## 4. Train both detectors and select using validation

The fixed settings are input 640×640, batch 8, seed 42, AdamW LR 0.001, up to 30 epochs,
patience 8 and FP32. Do not change settings after viewing test results. Training time depends
on the assigned GPU; use the printed epoch progress. The program reuses completed runs and
can resume an unfinished `last.pt` in the same work folder and environment.

AP comparison uses validation, confidence floor 0.001 and NMS IoU 0.70. The winning run then
sweeps validation confidence 0.01/0.025/0.05…0.95 and NMS IoU 0.45/0.60/0.75 at matching IoU 0.50.
The selected service threshold and low-floor AP metrics are saved separately.

Watch for BACKUP_CONFIRMED messages. An upload failure stops progression; fix it while retaining local files.


In [ ]:
# 4. Actual GPU training starts here. No test evaluation is performed.
from m4_train import run

RESULT, REVIEW_ZIP, MODELS_ZIP = run(
    input_root=INPUT_ROOT, work_root=WORK_ROOT, prepared_root=ROOT,
    wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY)


In [ ]:
# 5. Download both archives, and preserve the full notebook outputs for resuming.
from IPython.display import FileLink, Image as DisplayImage, display

display(FileLink(str(REVIEW_ZIP)))
display(FileLink(str(MODELS_ZIP)))
for filename in ['val_detector_comparison.png', 'val_threshold_sweep.png', 'val_selected_PR.png',
                 'train_val_learning_curves.png', 'val_development_gallery.png']:
    path = WORK_ROOT / 'figures' / filename
    if path.exists():
        display(DisplayImage(filename=str(path)))
print('SEND HERE:', REVIEW_ZIP.name)
print('KEEP FOR INTEGRATION:', MODELS_ZIP.name)
print('Test evaluation performed:', RESULT['test_evaluation_performed'])

print('Cloud backup receipt:')
print((WORK_ROOT / 'cloud_backup_receipt.json').read_text())


## Save the completed run

Wait for `BACKUP_CONFIRMED: M4_completed`, then download both ZIPs from cell 5 immediately.
Return the Results_Review ZIP; keep Models.zip for integration. Quick Save WITH output files enabled,
wait for saving, and verify the saved Output tab contains both ZIPs before ending the session.

The backup reference also appears in the printed receipt and W&B Artifacts tab. A future recovery
can restore this NEW run's checkpoint/metadata files by setting RESTORE_ARTIFACT in cell 2.
The prior notebook had no model-artifact upload, so this cannot recover its already lost weights.

Next: M5 segmentation on the same cohort, then M8 integration and one final test evaluation after
all settings are frozen. Do not rerun Phase 1 main training as part of this replacement.
CPU fixture checks passed; real cloud upload and GPU execution must be verified by the Kaggle run.
